# Lab: Find the planted bug
Run the buggy fetcher. Use breakpoint() + structlog to isolate the bug in under 10 minutes.

In [ ]:
!pip install -q httpx structlog

In [ ]:
import asyncio, httpx, structlog
structlog.configure(processors=[
    structlog.contextvars.merge_contextvars,
    structlog.processors.add_log_level,
    structlog.processors.JSONRenderer(),
])
log = structlog.get_logger()
HN = 'https://hacker-news.firebaseio.com/v0'

async def fetch_title_buggy(c, i):
    r = await c.get(f'{HN}/item/{i}.json', timeout=10)
    item = r.json()
    return item['title']  # bug: item can be None; title can be missing

async def run():
    async with httpx.AsyncClient() as c:
        ids = list(range(1, 11))
        for i in ids:
            structlog.contextvars.bind_contextvars(item_id=i)
            try:
                t = await fetch_title_buggy(c, i)
                log.info('ok', title=t[:40])
            except Exception as e:
                log.error('boom', err=str(e))

await run()

In [ ]:
# Fix: guard against None and missing keys.
async def fetch_title(c, i):
    r = await c.get(f'{HN}/item/{i}.json', timeout=10)
    item = r.json()
    if not item:
        return None
    return item.get('title')

async def run_fixed():
    async with httpx.AsyncClient() as c:
        return [t for t in [await fetch_title(c, i) for i in range(1, 11)] if t]

titles = await run_fixed()
print(len(titles), 'titles')

No LLM call. cost = $0.00 / run.